# Calendar generálás

Ez a notebook a silver calendar réteg generálására szolgál.

A calendar réteg célja, hogy assetenként és hónaponként legyen egy referencia időrács, amihez később a brókerekből érkező bronze OHLCV adatok lefedettsége hasonlítható.

A BTCUSD calendar 24/7 perces rácsként készül,  a kripto piac folyamatosan kereskedáse végett.  
A generált calendar a silver rétegbe kerül:

`data/silver/calendar/btcusd/{year}/{month}/`

A futás minden hónapra létrehozza:
- a calendar parquet fájlt;
- a `_MANIFEST.json` fájlt;
- a `_SUCCESS` markert.

Ha `START_MONTH` és `END_MONTH` értéke `None`, akkor a futás a `config/download_period.csv` szerinti időszakot használja.  
Ha megadjuk őket `YYYY-MM` formátumban, akkor csak az adott teljes, lezárt hónapok kerülnek feldolgozásra.

A nem kripto instrumentek calendarje később Interactive Brokers historical schedule alapján készül.


In [2]:
from pipelines.calendar_24_7_runner import run_24_7_calendar_from_config

import pandas as pd

# Ha mindkettő None, akkor a config/download_period.csv szerinti időszak fut.
# Ha megadod őket, csak a megadott teljes, lezárt hónapok futnak.
START_MONTH = None
END_MONTH = None

# Példa explicit időszakra:
# START_MONTH = "2024-01"
# END_MONTH = "2025-12"

results = run_24_7_calendar_from_config(
    asset="BTCUSD",
    start_month=START_MONTH,
    end_month=END_MONTH,
    interval="1m",
)

pd.DataFrame([result.__dict__ for result in results])


,status,asset,year,month,message,row_count
0,uploaded,BTCUSD,2024,1,24/7 calendar uploaded successfully.,44640
1,uploaded,BTCUSD,2024,2,24/7 calendar uploaded successfully.,41760
2,uploaded,BTCUSD,2024,3,24/7 calendar uploaded successfully.,44640
3,uploaded,BTCUSD,2024,4,24/7 calendar uploaded successfully.,43200
4,uploaded,BTCUSD,2024,5,24/7 calendar uploaded successfully.,44640
5,uploaded,BTCUSD,2024,6,24/7 calendar uploaded successfully.,43200
6,uploaded,BTCUSD,2024,7,24/7 calendar uploaded successfully.,44640
7,uploaded,BTCUSD,2024,8,24/7 calendar uploaded successfully.,44640
8,uploaded,BTCUSD,2024,9,24/7 calendar uploaded successfully.,43200
9,uploaded,BTCUSD,2024,10,24/7 calendar uploaded successfully.,44640


## Interactive Brokers alapú silver calendar generálás

Ez a cella az Interactive Brokers historical schedule adatai alapján készít silver szintű perces kereskedési naptárat.

A naptár azt jelöli, hogy az adott assetnél mely percekben várunk adatot. Ez később a silver rétegben használható arra, hogy megmérjük a brókerenkénti lefedettséget, hiányzó perceket és az adatsor minőségét.

Ha `START_MONTH` és `END_MONTH` értéke `None`, akkor a futás a `config/download_period.csv` szerinti időszakot használja.  
Ha megadjuk őket `YYYY-MM` formátumban, akkor csak az adott teljes, lezárt hónapok kerülnek feldolgozásra.

Az `ASSETS` paraméterrel szabályozható, mely assetekre készüljön naptár:
- `None`: minden IB calendar configban szereplő asset
- `["DAX"]`: csak DAX
- `["XAUUSD", "XAGUSD", "EURUSD", "US500", "DAX"]`: explicit lista

A BTCUSD nem innen készül, mert az 24/7 piac. Arra a külön 24/7 calendar generátor használható.

Fontos: futtatás előtt az Interactive Brokers TWS legyen elindítva, és az API kapcsolat legyen engedélyezve.


In [3]:
from ib_insync import util
util.startLoop()

import pandas as pd

from pipelines.calendar_interactive_brokers_runner import (
    run_interactive_brokers_calendar_from_config,
)


# Ha mindkettő None, akkor a config/download_period.csv szerinti időszak fut.
# Ha megadod őket, csak a megadott teljes, lezárt hónapok futnak.
START_MONTH = None
END_MONTH = None

# Példa explicit időszakra:
# START_MONTH = "2024-01"
# END_MONTH = "2024-12"

# Futtatandó assetek.
# None = minden IB calendar configban szereplő asset
# ["DAX"] = csak DAX
# ["XAUUSD", "XAGUSD", "EURUSD", "US500", "DAX"] = explicit lista
ASSETS = None

results = run_interactive_brokers_calendar_from_config(
    start_month=START_MONTH,
    end_month=END_MONTH,
    assets=ASSETS,
    interval="1m",
    num_days=45,
    use_regular_trading_hours=False,
    request_sleep_sec=1,
    print_progress=True,
)

pd.DataFrame([result.__dict__ for result in results])

XAUUSD 2024-01 calendar downloading...
XAGUSD 2024-01 calendar downloading...
EURUSD 2024-01 calendar downloading...
US500 2024-01 calendar downloading...
XAUUSD 2024-02 calendar downloading...
XAGUSD 2024-02 calendar downloading...
EURUSD 2024-02 calendar downloading...
US500 2024-02 calendar downloading...
DAX 2024-02 calendar downloading...
XAUUSD 2024-03 calendar downloading...
XAGUSD 2024-03 calendar downloading...
EURUSD 2024-03 calendar downloading...
US500 2024-03 calendar downloading...
DAX 2024-03 calendar downloading...
XAUUSD 2024-04 calendar downloading...
XAGUSD 2024-04 calendar downloading...
EURUSD 2024-04 calendar downloading...
US500 2024-04 calendar downloading...
DAX 2024-04 calendar downloading...
XAUUSD 2024-05 calendar downloading...
XAGUSD 2024-05 calendar downloading...
EURUSD 2024-05 calendar downloading...
US500 2024-05 calendar downloading...
DAX 2024-05 calendar downloading...
XAUUSD 2024-06 calendar downloading...
XAGUSD 2024-06 calendar downloading...
EUR

,status,asset,year,month,message,row_count
0,uploaded,XAUUSD,2024,1,IB calendar uploaded successfully.,31740
1,uploaded,XAGUSD,2024,1,IB calendar uploaded successfully.,31740
2,uploaded,EURUSD,2024,1,IB calendar uploaded successfully.,31650
3,uploaded,US500,2024,1,IB calendar uploaded successfully.,29040
4,skipped,DAX,2024,1,_SUCCESS already exists in Azure.,0
...,...,...,...,...,...,...
115,uploaded,XAUUSD,2025,12,IB calendar uploaded successfully.,31740
116,uploaded,XAGUSD,2025,12,IB calendar uploaded successfully.,31740
117,uploaded,EURUSD,2025,12,IB calendar uploaded successfully.,30840
118,uploaded,US500,2025,12,IB calendar uploaded successfully.,30135
